In [ ]:
import pandas as pd
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

In [ ]:
from src.data_loader import load_dataset_consolide
df = load_dataset_consolide()

On cherche à répondre à la question suiva,te: "Dans quelle mesure peut-on prédire, pour chaque station de mesure, le dépassement du seuil réglementaire de PM2.5 (25 µg/m³) dans les 24 prochaines heures à partir d’observations de pollution atmosphérique, de variables météorologiques, de données de trafic routier et des sorties d’un modèle physique atmosphérique ?"

Le modèle est estimé sur l’ensemble des stations afin de mutualiser l’information disponible. Les performances sont ensuite évaluées à la fois globalement et de manière désagrégée par station. Cette approche permet d’identifier d’éventuelles disparités spatiales dans la qualité des prédictions et d’évaluer la robustesse du modèle à l’échelle locale.

On doit d'abord enlever les valeurs manquantes pour éviter que le modèle plante

In [ ]:
df = df.dropna()

On trie ensuite les colonnes en fonction de la date

In [ ]:
df = df.sort_values("datetime_debut")
df.columns = df.columns.str.strip()

Certaines variables inutiles doivent être supprimer avant l'entrainement du modèle, sinon le modèle entrainera du bruit

In [ ]:
df = df.drop(columns=["nom_station", "type_station", "lon", "lat", "distance_km", "datetime_debut"])

La variable 'saison' étant utile à notre étude d'après les résultats de la partie descriptives, on l'encode afin de l'intégrer au modèle

In [ ]:
# df = pd.get_dummies(df, columns=["saison"])

On définit ensuite le vecteur des features à utiliser dans notre modèle

In [ ]:
features = [
    "pm25_lag1h",
    "pm25_lag6h",
    "pm25_lag24h",
    "pm25_roll24h",
    "pm25_roll72h",
    "vent_vitesse_ms",
    "vent_direction_deg",
    "temperature_c",
    "humidite_pct",
    "pluie_mm",
    "nb_installations_5km",
    "heure",
    "jour_semaine",
    "mois",
    "is_weekend"
]

target = "depasse_seuil_24h"

In [ ]:
from src.models import prepare_features, temporal_split, evaluate_model, train_cart, train_random_forest, train_logistic_regression

X,y = prepare_features(df, features, target)
X_train, X_test, y_train, y_test = temporal_split(X, y, train_size=0.8)

On en entraine alors successivement les modèles CART, forêt aléatoire et régression logitique

In [ ]:
model_cart = train_cart(X_train, y_train, max_depth=3, min_samples_leaf=50, random_state=42)
model_rf = train_random_forest(X_train, y_train, n_estimators=100, max_depth=5, min_samples_leaf=50, random_state=42)
model_log = train_logistic_regression(X_train, y_train, X_test)

On peut visualiser l'arbre

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 10))
plot_tree(
    model_cart,
    feature_names=X_train.columns,
    class_names=["0", "1"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.show()